# Capstone · Notebook 2. Post-training: SFT and DPO

> **Files for this notebook.** Download [course-materials.zip](https://vladkulikov.com/downloads/modern-llms/course-materials.zip) from the course website and extract the complete archive, preserving the `capstone` folder. In Colab, upload the same ZIP when the notebook prompts for files.

*Modern LLMs course · capstone · edition 2026.10.3*

> **What happens here.** The base model from notebook 1 can continue text but
> cannot hold a conversation. We teach it the format — chat template, loss
> masking over assistant replies, packing (M13) — and then shift its preferences
> with DPO (M14).
>
> **The central experiment is §5.** We deliberately leave the closing
> `<|eot_id|>` out of the trainable part of the mask and watch what happens.
> We measure the changed training objective and both branches’ behavior.
>
> **The Colab contract.** The notebook is self-contained: if no checkpoint from
> notebook 1 is nearby, it trains a small base model itself. No network required.

A useful working distinction is that SFT demonstrates desired behaviour while
preference or reinforcement learning supplies a different training signal. At this scale we verify the mechanics, not frontier-level answer quality.

On a first pass, run the cells in order and connect the code to the explanations. On a second, change one setting, retain the results, and state which claim the comparison can test.

## 1. Environment and the base model

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for package_root in (search_root / "capstone" / "minillm" / "en", search_root / "minillm" / "en"):
        found = _activate_capstone_root(package_root)
        if found:
            return found
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "project package: " + found

    try:
        import minillm  # noqa: F401
        return "minillm is already importable"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB)) + sorted(Path.cwd().glob("course-materials.zip"))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Upload course-materials.zip from the course notebook page. The directory structure will be restored automatically.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB)) + sorted(Path.cwd().glob("course-materials.zip"))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        root = _find_capstone_root(destination)
        if root:
            return "project archive: " + str(archive_path)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "project archive: " + str(archive_path)

    raise RuntimeError("minillm was not found. Download course-materials.zip from the course notebook page and either extract it in full or upload the ZIP to the current Colab session.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import os, torch, numpy as np, random

# Run profile. SMOKE is sized for CPU and finishes in minutes; FULL only makes
# sense on a GPU and produces noticeably more coherent answers.
PROFILE = "SMOKE"
BUDGET = {
    "SMOKE": dict(base_steps=250, sft_steps=200, sft_batch=4, seq_len=192,
                  dpo_steps=100, dpo_batch=4, hidden=128, layers=3, dialogues=800),
    "FULL":  dict(base_steps=2000, sft_steps=1500, sft_batch=8, seq_len=256,
                  dpo_steps=600, dpo_batch=8, hidden=384, layers=6, dialogues=6000),
}[PROFILE]
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    BUDGET = {**BUDGET, "base_steps": min(BUDGET["base_steps"], 10),
              "sft_steps": min(BUDGET["sft_steps"], 12),
              "dpo_steps": min(BUDGET["dpo_steps"], 8),
              "dialogues": min(BUDGET["dialogues"], 80),
              "sft_batch": min(BUDGET["sft_batch"], 2),
              "dpo_batch": min(BUDGET["dpo_batch"], 2)}
SEED = 0
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

The base model. If a checkpoint from notebook 1 sits nearby we load it; otherwise
we train a small base model right here. The second path is faster and sufficient
for every conclusion in this notebook: what interests us is **what the mask
teaches**, not absolute quality.

In [ ]:
from minillm.chat import CHAT_SPECIALS
from minillm.config import MiniLLMConfig
from minillm.data import TokenStream
from minillm.model import MiniLLM
from minillm.tokenizer import BBPETokenizer
from minillm.train import TrainConfig, train

EN_SLOTS = dict(
    subject=["the language model", "the transformer", "the attention mechanism",
             "the tokenizer", "the optimizer", "normalization", "the residual stream",
             "the expert router"],
    verb=["predicts", "processes", "reduces", "accelerates", "stabilizes"],
    obj=["the next token", "a long context", "the cost per step", "the memory footprint",
         "the scale of activations", "the load across experts"],
)

def sentence(rng, slots):
    return " ".join([rng.choice(slots["subject"]), rng.choice(slots["verb"]),
                     rng.choice(slots["obj"])]).capitalize() + "."

rng = random.Random(SEED)
base_text = "\n".join(" ".join(sentence(rng, EN_SLOTS) for _ in range(4))
                      for _ in range(4000))

def usable_checkpoint(path: Path):
    """A checkpoint is usable only if its vocabulary knows the chat control tokens.

    A vocabulary without `<|eot_id|>` would crash the notebook on the very first
    loss mask, and with an unhelpful KeyError at that. Notebook 1 reserves those
    identifiers up front, but a checkpoint may have been built by an older
    version — so we check explicitly.
    """
    if not (path / "config.json").exists():
        return None, None, "no notebook-1 checkpoint nearby"
    tok = BBPETokenizer.load(path / "tokenizer.json")
    missing = [s for s in CHAT_SPECIALS if s not in tok.specials]
    if missing:
        return None, None, f"the checkpoint vocabulary lacks {missing}"
    from minillm.train import load_checkpoint
    model, _, _, _ = load_checkpoint(path, device=DEVICE)
    return model, tok, None

CHECKPOINT = Path("runs/pretrain/final")
base_model, tokenizer, reason = usable_checkpoint(CHECKPOINT)
if base_model is not None:
    print("loaded the checkpoint from notebook 1:", base_model.num_parameters(), "parameters")
else:
    print(f"training a base model ourselves: {reason}")
    tokenizer = BBPETokenizer.train(base_text, vocab_size=1024, specials=CHAT_SPECIALS)
    cfg = MiniLLMConfig.profile("S", vocab_size=tokenizer.vocab_size,
                                hidden_size=BUDGET["hidden"],
                                num_hidden_layers=BUDGET["layers"],
                                num_attention_heads=4, num_key_value_heads=2,
                                head_dim=32, max_position_embeddings=BUDGET["seq_len"])
    base_model = MiniLLM(cfg).to(DEVICE)
    stream = TokenStream.from_text(base_text, tokenizer, seq_len=128)
    print(f"parameters: {base_model.num_parameters():,}")
    train(base_model, stream, TrainConfig(steps=BUDGET["base_steps"], seq_len=128,
                                          micro_batch_size=8, optimizer="muon",
                                          seed=SEED, device=DEVICE, log_every=125))
print("vocabulary:", tokenizer.vocab_size)

## 2. The instruction dataset

The dialogue generator reuses the pretraining slot grammar, reducing changes in vocabulary and topic. That does not establish that SFT learns format alone. Random slot combinations are teaching fixtures and may be factually nonsensical. For the EOT experiment, both branches receive the same dialogues, isolating the mask change rather than assigning every SFT effect to formatting.

In [ ]:
from minillm.chat import ChatTemplate, Message, SFTDataset

QUESTIONS = [
    ("What does {s} do?", "{s} {v} {o}."),
    ("What is {s} used for?", "{s} {v} {o}. That is its main role here."),
    ("How does {s} affect the model?", "{s} {v} {o}, changing the model behaviour."),
]

def make_dialogues(n: int, seed: int = 0) -> list[list[Message]]:
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        question, answer = rng.choice(QUESTIONS)
        s = rng.choice(EN_SLOTS["subject"])
        v, o = rng.choice(EN_SLOTS["verb"]), rng.choice(EN_SLOTS["obj"])
        out.append([
            Message("system", "Answer briefly and directly."),
            Message("user", question.format(s=s)),
            Message("assistant", answer.format(s=s.capitalize(), v=v, o=o)),
        ])
    return out

template = ChatTemplate()
dialogues = make_dialogues(BUDGET["dialogues"], seed=SEED)
print(template.render(dialogues[0], add_generation_prompt=False))

## 3. The mask: what exactly the model learns

The rule of M13: train on assistant replies **and the closing `<|eot_id|>`**;
do not train on the system prompt, the user turns or the role headers — those are
written by the runtime.

In [ ]:
EOT_ID = tokenizer.specials["<|eot_id|>"]
PAD_ID = tokenizer.specials["<|begin_of_text|>"]

ids, mask = template.encode_with_mask(dialogues[0], tokenizer)
print(f"tokens: {len(ids)}, trainable: {sum(mask)} "
      f"({template.trainable_fraction(mask):.1%})\n")

print(f"{'token':<22} {'id':>6}  train")
for token_id, flag in list(zip(ids, mask)):
    piece = tokenizer.decode([token_id]).replace("\n", "\\n")
    print(f"{piece!r:<22} {token_id:>6}  {'YES' if flag else '—'}")

In [ ]:
dataset, stats = SFTDataset.from_dialogues(dialogues, tokenizer, template,
                                           seq_len=BUDGET["seq_len"], pad_id=PAD_ID)
print("packing:", stats)
print(f"\nwithout packing this would need {stats['dialogues']} rows of "
      f"{BUDGET['seq_len']} tokens,")
print(f"with packing — {stats['bins']}: a ×{stats['packing_ratio']} saving, "
      f"padding {stats['padding_fraction']:.1%}")

Packing here is a correctness feature as well as an efficiency feature. The
dataset returns `segment_ids` beside `ids` and `labels`; the model builds a
block-diagonal causal mask and resets RoPE positions inside every dialogue.
Changing one packed example therefore cannot alter the logits of its neighbour.
`<|eot_id|>` marks the protocol boundary, but it would not provide this isolation
by itself.

## 4. SFT

We train the same model with the same loop — only the source of batches changes:
instead of a token stream it now yields `(ids, labels, segment_ids)`. The −100
labels disable direct loss, while segment IDs build a block-diagonal causal mask
and reset positions inside every independently packed dialogue.


SFT and validation use a mean over supervised targets after the causal shift. With gradient accumulation, each microbatch contributes in proportion to its target count. Repartitioning the same examples therefore preserves the chosen loss objective.

In [ ]:
import copy

sft_model = copy.deepcopy(base_model)
sft_cfg = TrainConfig(steps=BUDGET["sft_steps"], seq_len=BUDGET["seq_len"],
                      micro_batch_size=BUDGET["sft_batch"], optimizer="muon",
                      seed=SEED, device=DEVICE, log_every=BUDGET["sft_steps"] // 4,
                      out_dir="runs/sft")
sft_state = train(sft_model, dataset, sft_cfg)

In [ ]:
from minillm.generate import SamplingConfig, generate

def ask(model, question: str, max_new_tokens: int = 60, temperature: float = 0.0) -> tuple[str, int]:
    ids, _ = template.encode_with_mask(
        [Message("system", "Answer briefly and directly."),
         Message("user", question)], tokenizer, add_generation_prompt=True)
    out, stats = generate(model, ids, max_new_tokens,
                          SamplingConfig(temperature=temperature, seed=SEED),
                          stop_tokens={EOT_ID})
    answer = tokenizer.decode(out[len(ids):])
    return answer, stats.generated_tokens

question = "What does the tokenizer do?"
for label, model in (("before SFT", base_model), ("after SFT", sft_model)):
    answer, n = ask(model, question)
    print(f"--- {label} ({n} tokens)\n{answer!r}\n")

## 5. The central experiment: the forgotten `<|eot_id|>`

We repeat SFT with exactly one change: `train_eot=False`. The data are the same,
the sequences are the same — **one position of the mask** differs per dialogue.

In [ ]:
broken_dataset, _ = SFTDataset.from_dialogues(dialogues, tokenizer, template,
                                              seq_len=BUDGET["seq_len"], pad_id=PAD_ID,
                                              train_eot=False)
a_ids, a_mask = template.encode_with_mask(dialogues[0], tokenizer, train_eot=True)
b_ids, b_mask = template.encode_with_mask(dialogues[0], tokenizer, train_eot=False)
diff = [i for i, (x, y) in enumerate(zip(a_mask, b_mask)) if x != y]
print(f"sequences identical: {a_ids == b_ids}")
print(f"masks differ at positions {diff} — that is {tokenizer.decode([a_ids[diff[0]]])!r}")

broken_model = copy.deepcopy(base_model)
train(broken_model, broken_dataset,
      TrainConfig(steps=BUDGET["sft_steps"], seq_len=BUDGET["seq_len"],
                  micro_batch_size=BUDGET["sft_batch"], optimizer="muon", seed=SEED,
                  device=DEVICE, log_every=BUDGET["sft_steps"], out_dir="runs/sft_broken"))

In [ ]:
eot_rows = []

print(f"{'model':<16} {'p(<|eot_id|>) after reply':>28} {'reply length':>14}")
for label, model in (("mask correct", sft_model), ("EOT forgotten", broken_model)):
    ids, _ = template.encode_with_mask(dialogues[0], tokenizer)
    prefix = ids[:len(ids) - 1]        # everything but the closing EOT
    with torch.no_grad():
        logits, _ = model(torch.tensor([prefix], device=DEVICE))
        p_eot = torch.softmax(logits[0, -1].float(), dim=-1)[EOT_ID].item()
    _, generated = ask(model, "What does the tokenizer do?", max_new_tokens=120)
    eot_rows.append(dict(label=label, probability=p_eot, generated_tokens=generated))
    print(f"{label:<16} {p_eot:>28.4f} {generated:>14}")

answer, n = ask(broken_model, "What does the tokenizer do?", max_new_tokens=120)
print(f"\nreply from the model with the forgotten EOT ({n} tokens):\n{answer!r}")

# Use one correct mask for both models; the generator is a diagnostic fixture.
from minillm.train import evaluate
check_dialogues = make_dialogues(32, seed=SEED + 991)
check_dataset, _ = SFTDataset.from_dialogues(check_dialogues, tokenizer, template,
    seq_len=BUDGET["seq_len"], pad_id=PAD_ID, train_eot=True)
shared_scores = {name: evaluate(model, check_dataset, sft_cfg, batches=4)
                 for name, model in (("correct_mask", sft_model), ("omitted_eot", broken_model))}
assert sum(a_mask) == sum(b_mask) + len(diff)
print("EOT supervised positions:", sum(a_mask), "vs", sum(b_mask))
print("Shared correct-mask evaluation:", shared_scores)

Removing EOT changes both the summed loss and the number of supervised positions in its denominator. The training losses now refer to different objectives and need not agree. Validation using the same faulty mask can conceal the missing stopping target.

Score both models on one shared dataset with EOT supervision restored, then inspect EOT probability after a specified answer and the length of greedy generation. A token probability and an observed failure to stop are related but distinct measurements. A short run may produce only a weak contrast; report its actual size rather than assuming a dramatic failure.

## 6. Preference pairs

These pairs are authored by a template: there is no model generation or human annotation in `build_pairs`. The chosen answer fills a `QUESTIONS` template with slot values; the rejected answer appends two identical filler sentences. Both terminate with EOT.

The construction makes a preference for brevity inspectable. It lets us observe a DPO update, but it does not represent the diversity of human preferences. Studying policy-generated data would require a separate generation stage with a fixed model revision and independently assessed candidates.

In [ ]:
from minillm.preference import (DPOConfig, PreferencePair, freeze_reference, train_dpo)

FILLER = " This matters and should be considered when working with the model."

def build_pairs(n: int, seed: int = 0) -> list[PreferencePair]:
    rng = random.Random(seed)
    pairs = []
    for _ in range(n):
        question, answer = rng.choice(QUESTIONS)
        s = rng.choice(EN_SLOTS["subject"])
        v, o = rng.choice(EN_SLOTS["verb"]), rng.choice(EN_SLOTS["obj"])
        prompt_ids, _ = template.encode_with_mask(
            [Message("user", question.format(s=s))], tokenizer,
            add_generation_prompt=True)
        short = answer.format(s=s.capitalize(), v=v, o=o)
        verbose = short + FILLER * 2
        pairs.append(PreferencePair(
            prompt_ids=prompt_ids,
            chosen_ids=tokenizer.encode(short) + [EOT_ID],
            rejected_ids=tokenizer.encode(verbose) + [EOT_ID],
        ))
    return pairs

pairs = build_pairs(192, seed=SEED)
print(f"pairs: {len(pairs)}")
print("chosen  :", tokenizer.decode(pairs[0].chosen_ids)[:90])
print("rejected:", tokenizer.decode(pairs[0].rejected_ids)[:90])

## 7. DPO

A frozen SFT copy supplies reference probabilities. DPO compares changes in the two responses' log probabilities relative to that reference. After training, inference needs only the updated policy.

`margin` is the mean implicit-reward difference; `accuracy` counts pairs with a positive difference. A zero margin is not counted as success here. Read both alongside loss and individual responses. Neither 0.5 accuracy nor a rapid rise to 1.0 identifies a cause on its own: easy pairs, tie handling, limited learning, leakage, and overfitting require different checks.

In [ ]:
policy = copy.deepcopy(sft_model)
reference = freeze_reference(sft_model)
dpo_state = train_dpo(policy, pairs,
                      DPOConfig(beta=0.1, lr=1e-4, steps=BUDGET["dpo_steps"],
                                batch_size=BUDGET["dpo_batch"], seed=SEED,
                                device=DEVICE, log_every=BUDGET["dpo_steps"] // 8),
                      pad_id=PAD_ID, reference=reference)

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in dpo_state.history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(steps, [h["loss"] for h in dpo_state.history], color="#5E81AC")
axes[0].axhline(np.log(2), ls="--", lw=1, color="#4C566A", label="ln 2 — the start")
axes[0].set_title("DPO loss"); axes[0].legend()
axes[1].plot(steps, [h["margin"] for h in dpo_state.history], color="#A3BE8C")
axes[1].axhline(0, ls="--", lw=1, color="#4C566A")
axes[1].set_title("margin: difference of implicit rewards")
axes[2].plot(steps, [h["chosen_reward"] for h in dpo_state.history],
             label="chosen", color="#A3BE8C")
axes[2].plot(steps, [h["rejected_reward"] for h in dpo_state.history],
             label="rejected", color="#BF616A")
axes[2].set_title("implicit rewards"); axes[2].legend()
for ax in axes:
    ax.set_xlabel("step"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

last = dpo_state.history[-1]
print(f"margin {dpo_state.history[0]['margin']:+.3f} → {last['margin']:+.3f}, "
      f"accuracy {last['accuracy']:.2f}")

The third plot exposes the two terms behind the margin. DPO can increase their difference without increasing the chosen response's absolute probability. Implicit reward is not a direct utility measurement either. Downward movement in both terms warrants investigation, but does not by itself demonstrate degraded answers.

Length and repeated filler make these pairs easy to distinguish. Check how quickly this run separates them, then reduce the filler while keeping other settings fixed. Claims about quality require held-out pairs and content checks in addition to the training curves.

In [ ]:
question = "What is the optimizer used for?"
for label, model in (("after SFT", sft_model), ("after DPO", policy)):
    answer, n = ask(model, question, max_new_tokens=80, temperature=0.7)
    print(f"--- {label} ({n} tokens)\n{answer!r}\n")
print('Two responses are illustrations. Compare several fixed prompts and save outcomes before inferring a behavioral change.')

## 8. Saving

After DPO we save an **inference artifact**: weights, `config.json`, the
tokenizer, and JSON stage provenance. This is deliberately not a resumable
checkpoint. The DPO weights no longer correspond to the SFT optimizer moments,
and presenting those stale moments as resumable state would be a silent error.
The full resumable pretraining checkpoint remains in `runs/pretrain/final`.

In [ ]:
from minillm.train import save_model_artifact

out_dir = Path("runs/posttrain/final")
save_model_artifact(out_dir, policy, {
    "stage": "dpo",
    "sft_steps": sft_state.step,
    "dpo_steps": dpo_state.step,
    "note": "inference artifact; DPO optimizer state is intentionally not stored",
})
tokenizer.save(out_dir / "tokenizer.json")
print("saved:", sorted(p.name for p in out_dir.iterdir()))

import json
metrics = {"seed": SEED, "budget": BUDGET, "eot": eot_rows,
           "shared_mask_scores": shared_scores, "dpo_history": dpo_state.history}
(out_dir / "teaching_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

## 9. What the experiment establishes

The chat template, supervision mask, and packing rules define the SFT task. The EOT experiment changes the mask and compares both branches under a shared scoring rule. DPO uses explicit template preferences; the margin describes those particular pairs.

SFT can teach content as well as format. Preference learning changes behavior according to a signal whose quality must also be examined. The measurements here describe synthetic dialogues and authored pairs. Real-world answer utility and the effects of a separate RL stage require their own experiments and evaluation criteria.

Retain the actual loss, margin, EOT probabilities, and response lengths with their training budget. They describe this run on this corpus. Continue with inference in Notebook 3 and controlled architecture and training comparisons in Notebook 4.